# Module 1: Data Ingestion, Quality Auditing & Clinical Imputation
**Project #49:** Stroke Risk Factors & Clinical Triage App  
**Student:** Amrutha varshini (`S230997`) | **Course:** Introduction to Data Science (`25A12104`)

---

## Objectives
1. Audit raw dataset dimensions, data types, duplicate records, missing values, outliers, and severe target imbalance.
2. Evaluate whether missingness in `bmi` carries clinical signal (MNAR hypothesis: acutely ill stroke patients cannot stand for BMI measurement).
3. Address the single `gender == "Other"` record and document statistical justification.
4. Compare imputation strategies for `bmi` (Stratified Age-Band & Gender Median vs. Global Median) and `smoking_status == "Unknown"` (pediatric `<12` imputation + adult `"keep"` vs. stratified `"mode"` CV ROC-AUC benchmark).
5. Export the verified exploratory dataset to `data/processed/stroke_cleaned_eda.csv`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.config import BMI_CAP, CLEANED_EDA_PATH, CV_FOLDS, DATA_RAW_PATH, RANDOM_STATE
from src.data_cleaning import (
    ClinicalDataCleaner,
    audit_raw_data,
    clean_for_eda,
    drop_other_gender_rows,
    load_raw_data,
)

raw_df = load_raw_data(DATA_RAW_PATH)
print(f"Raw dataset shape: {raw_df.shape}")
raw_df.head()

Raw dataset shape: (5110, 12)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## 1. Raw Data Quality Audit

In [2]:
audit = audit_raw_data(raw_df)
for k, v in audit.items():
    print(f"{k}: {v}")

print("\nData types:")
print(raw_df.dtypes)

n_rows: 5110
n_cols: 12
duplicate_rows: 0
unique_ids: 5110
missing_counts: {'bmi': 201}
gender_counts: {'Female': 2994, 'Male': 2115, 'Other': 1}
stroke_counts: {0: 4861, 1: 249}
stroke_rate: 0.0487279843444227
bmi_outliers_gt_60: 13
bmi_max: 97.6
stroke_rate_when_bmi_present: 0.04257486249745366
stroke_rate_when_bmi_missing: 0.19900497512437812
smoking_unknown_total: 1544
smoking_unknown_pediatric_lt12: 521
pediatric_lt12_total: 543
smoking_unknown_adult_ge12: 1023

Data types:
id                     int64
gender                   str
age                  float64
hypertension           int64
heart_disease          int64
ever_married             str
work_type                str
Residence_type           str
avg_glucose_level    float64
bmi                  float64
smoking_status           str
stroke                 int64
dtype: object


### Key Audit Insights (Viva Takeaways)
- **Target Class Imbalance:** Out of 5,110 records, only **249 (4.87%)** are positive stroke cases vs. **4,861 (95.13%)** non-stroke cases.
- **Informative Missingness in `bmi`:** Missing `bmi` occurs in 201 rows (3.93%). Crucially, the stroke rate among patients **with missing BMI is 19.90% (40/201)** compared to **4.26% (209/4,909)** when BMI is recorded — a **4.7x higher stroke prevalence**! Therefore, we create `bmi_was_missing` before imputation so models do not lose this acute-admission signal.
- **Rare `gender == "Other"` Record:** Only 1 row (`id = 56156`, age 26, `stroke = 0`, 0.02%). Dropping this single row prevents near-zero-variance sparse dummy issues in Chi-Square (`expected < 5`) and Logistic Regression quasi-separation.
- **BMI Outliers (`> 60`):** 13 records have `bmi > 60` (max `97.6`), all with `stroke = 0`. Winsorizing implausible extremes above `65.0` (only 5 records) protects linear/distance models from extreme leverage without deleting valid patients.

## 2. Smoking Status Audit & Pediatric vs. Adult Imputation Experiment

In [3]:
filtered_df = drop_other_gender_rows(raw_df)

def eval_smoking_strategy(strategy: str, bmi_cap: float | None = BMI_CAP) -> dict:
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    X = filtered_df.drop(columns=["stroke"])
    y = filtered_df["stroke"]
    num_cols = ["age", "avg_glucose_level", "bmi"]
    bin_cols = ["hypertension", "heart_disease", "bmi_was_missing", "smoking_was_unknown"]
    cat_cols = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]

    pre = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("bin", "passthrough", bin_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols),
    ])
    pipe = Pipeline([
        ("cleaner", ClinicalDataCleaner(bmi_cap=bmi_cap, adult_unknown_strategy=strategy)),
        ("pre", pre),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ])
    roc = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc").mean()
    pr = cross_val_score(pipe, X, y, cv=cv, scoring="average_precision").mean()
    return {"strategy": strategy, "bmi_cap": bmi_cap, "cv_roc_auc": round(roc, 4), "cv_pr_auc": round(pr, 4)}

exp_df = pd.DataFrame([
    eval_smoking_strategy("keep", BMI_CAP),
    eval_smoking_strategy("mode", BMI_CAP),
    eval_smoking_strategy("keep", None),
])
exp_df

,strategy,bmi_cap,cv_roc_auc,cv_pr_auc
0,keep,65.0,0.8468,0.2234
1,mode,65.0,0.8469,0.2218
2,keep,NaN,0.8469,0.2232


### Imputation Decision Rationale
- **Pediatric `Unknown` (`age < 12`):** 521 out of 543 children (`95.9%`) have `smoking_status == "Unknown"`. Imputing these pediatric cases to `"never smoked"` is clinically sound and removes pediatric confounding from the `Unknown` category.
- **Adult `Unknown` (`age >= 12`):** Retaining `"Unknown"` as an explicit category alongside `smoking_was_unknown` achieves equivalent or higher CV ROC-AUC/PR-AUC without fabricating unverified smoking histories for 1,023 adult patients.
- **BMI Imputation & Capping:** Stratified Age-Band & Gender median imputation preserves demographic BMI differences, and capping implausible extremes at `65.0` maintains identical discrimination while preventing extreme leverage.

## 3. Generate & Verify Cleaned EDA Dataset

In [4]:
cleaned_df, _ = clean_for_eda(DATA_RAW_PATH, CLEANED_EDA_PATH, adult_unknown_strategy="keep")
assert cleaned_df.isna().sum().sum() == 0, "Unexpected NaNs remain!"
assert cleaned_df.shape == (5109, 13), f"Unexpected shape: {cleaned_df.shape}"
print("Saved cleaned EDA dataset to:", CLEANED_EDA_PATH)
print("Shape:", cleaned_df.shape)
print("\nSummary statistics (numeric):")
display(cleaned_df[["age", "avg_glucose_level", "bmi", "bmi_was_missing", "smoking_was_unknown", "stroke"]].describe().round(3))
print("\nSmoking status counts after pediatric imputation:")
print(cleaned_df["smoking_status"].value_counts())

Saved cleaned EDA dataset to: /home/amrutha/Stroke-Risk-Project/stroke-risk-triage-app/data/processed/stroke_cleaned_eda.csv
Shape: (5109, 13)

Summary statistics (numeric):


,age,avg_glucose_level,bmi,bmi_was_missing,smoking_was_unknown,stroke
count,5109.000,5109.000,5109.000,5109.000,5109.000,5109.000
mean,43.230,106.140,28.855,0.039,0.302,0.049
std,22.614,45.285,7.621,0.194,0.459,0.215
min,0.080,55.120,10.300,0.000,0.000,0.000
25%,25.000,77.240,23.700,0.000,0.000,0.000
50%,45.000,91.880,28.300,0.000,0.000,0.000
75%,61.000,114.090,32.800,0.000,1.000,0.000
max,82.000,271.740,65.000,1.000,1.000,1.000



Smoking status counts after pediatric imputation:
smoking_status
never smoked       2413
Unknown            1023
formerly smoked     884
smokes              789
Name: count, dtype: int64
